# 📓 Semana 1 · Dia 5 — SQL analítico profundo e criação das tabelas Bronze

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (ELT with Spark SQL) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Tabelas `vendas_bronze` e `voos_bronze` criadas |

---


## 📖 Teoria — Padrões SQL que caem na prova

**HAVING** filtra o resultado de uma agregação (diferente de WHERE, que filtra antes):
```sql
SELECT Country, COUNT(*) n FROM vendas GROUP BY Country HAVING n > 100
```
**CASE WHEN** cria colunas condicionais (essencial para faixas, flag de devolução):
```sql
CASE WHEN Quantity < 0 THEN 'devolução' ELSE 'venda' END
```
**JOINs**: inner, left, right, full, semi (existe na esquerda), anti (não existe na esquerda).
**CTE (WITH)**: legibilidade para pipelines complexos.

> 🎯 **Dica de prova**: SEMI e ANTI joins caem direto. Semi = filtro; Anti = exclusão. Memorize: `LEFT SEMI` devolve só colunas da esquerda; `LEFT ANTI` devolve linhas da esquerda sem correspondência na direita.


### 💻 Na prática — Criando as tabelas Bronze

Agora criamos as tabelas governadas pelo UC com tipo `DELTA` — a base da arquitetura Medallion. Adicionamos `_ingested_at` (quando o dado chegou) — padrão de Bronze.


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE OR REPLACE TABLE workspace.bronze.vendas_bronze (
  InvoiceNo      STRING,
  StockCode      STRING,
  Description    STRING,
  Quantity       INT,
  InvoiceDate    TIMESTAMP,
  UnitPrice      DOUBLE,
  CustomerID     STRING,
  Country        STRING,
  _ingested_at   TIMESTAMP DEFAULT current_timestamp()
) USING DELTA;

In [ ]:
# Carregar os dados limpos do CSV para a tabela Delta
from pyspark.sql.functions import to_timestamp
df_vendas = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .load("/Volumes/workspace/bronze/vol_dados_curso/vendas.csv")
df_limpo = (df_vendas
    .withColumn("InvoiceDate", to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    .filter("Quantity > 0 AND UnitPrice > 0"))
df_limpo.write.mode("overwrite").saveAsTable("workspace.bronze.vendas_bronze")
print("vendas_bronze:", df_limpo.count(), "linhas")

### 💻 Na prática — Segunda tabela: voos (dados secundários)

Criamos dados sintéticos de voos para enriquecer o projeto (frequência de viagem dos clientes). Será usado no RAG na Semana 11.


In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.voos_bronze (
  CustomerID   STRING,
  Voo          STRING,
  Origem       STRING,
  Destino      STRING,
  Data_Voo     TIMESTAMP,
  Classe       STRING,
  Valor_Bilhete DOUBLE,
  _ingested_at TIMESTAMP DEFAULT current_timestamp()
) USING DELTA;

In [ ]:
# Gerar dados sintéticos de voos para clientes do varejo
from datetime import datetime, timedelta
import random
random.seed(42)
clientes = [r["CustomerID"] for r in spark.sql(
    "SELECT DISTINCT CustomerID FROM workspace.bronze.vendas_bronze WHERE CustomerID IS NOT NULL").collect()]
aeroportos = ["GRU", "CGH", "GIG", "BSB", "POA", "CNF", "REC", "SSA", "CWB", "FLN"]
classes = ["Econômica", "Econômica", "Executiva", "Primeira"]
voos = []
for i, c in enumerate(clientes[:200]):
    for _ in range(random.randint(1, 4)):
        voos.append((c, f"VOO-{random.randint(1000,9999)}", random.choice(aeroportos),
                     random.choice(aeroportos),
                     datetime(2024,1,1) + timedelta(days=random.randint(0,365)),
                     random.choice(classes),
                     round(random.uniform(150, 2500), 2)))
df_voos = spark.createDataFrame(voos, ["CustomerID","Voo","Origem","Destino","Data_Voo","Classe","Valor_Bilhete"])
df_voos.write.mode("overwrite").saveAsTable("workspace.bronze.voos_bronze")
print(f"voos_bronze: {df_voos.count()} registros")

In [ ]:
%sql
-- Conferindo o que foi criado
SHOW TABLES IN workspace.bronze;
SELECT * FROM workspace.bronze.vendas_bronze LIMIT 5;

### 💻 Na prática — SQL analítico sobre as tabelas Bronze

Agora as queries rodam sobre a tabela governada (não mais sobre view temporária).


In [ ]:
%sql
-- HAVING: países com mais de 2.000 vendas
SELECT Country, COUNT(*) AS n FROM workspace.bronze.vendas_bronze
GROUP BY Country HAVING n > 2000 ORDER BY n DESC

In [ ]:
%sql
-- CASE WHEN: flag de ticket alto
SELECT Country,
       COUNT(*) AS vendas,
       SUM(CASE WHEN Quantity * UnitPrice > 50 THEN 1 ELSE 0 END) AS vendas_caras
FROM workspace.bronze.vendas_bronze
GROUP BY Country ORDER BY vendas DESC LIMIT 5

> 🎯 **Dica de prova**: `DATE_TRUNC`, `CASE WHEN`, `HAVING` e os 6 tipos de JOIN são perguntas garantidas. Pratique escrevê-los de memória.


## 🎯 Exercícios de fixação

**1.** Crie uma tabela `workspace.bronze.clientes_bronze` com dados sintéticos de clientes (CustomerID, Nome, Email, Cidade, Pais).

**2.** Escreva uma query com CTE que calcula receita por país e filtra países com receita > 100 mil.

**3.** Use LEFT ANTI para achar clientes que aparecem em vendas mas não na tabela de clientes.

**4.** O que `_ingested_at` representa e por que é padrão em camadas Bronze?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** clientes_bronze

```sql
CREATE OR REPLACE TABLE workspace.bronze.clientes_bronze (CustomerID STRING, Nome STRING, Email STRING, Cidade STRING, Pais STRING) USING DELTA;
```
Depois insira ~50 linhas sintéticas com Python (spark.createDataFrame).

**2.** CTE com HAVING equivalente

```sql
WITH receita_pais AS (
  SELECT Country, SUM(Quantity*UnitPrice) receita
  FROM workspace.bronze.vendas_bronze GROUP BY Country)
SELECT * FROM receita_pais WHERE receita > 100000 ORDER BY receita DESC;
```

**3.** LEFT ANTI

```sql
SELECT DISTINCT v.CustomerID FROM workspace.bronze.vendas_bronze v
LEFT ANTI JOIN workspace.bronze.clientes_bronze c ON v.CustomerID = c.CustomerID;
```

**4.** _ingested_at

Marca o momento em que a linha entrou no Bronze — permite reprocessamento, auditoria e janelas de ingestão. É o padrão de Bronze (append-only com timestamp de chegada).



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*